<a href="https://colab.research.google.com/github/ubaid8878/Flyrank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ubaid8878/Flyrank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
## 1. Unit of analysis + time window

For my Refresh / Content Opportunity Scoring lane, one row represents one content-page observation for a specific month.

I will work on a mid-panel month, `2026-03`, rather than the final month. The final month is treated as a sealed test period because future outcome information can otherwise leak into development decisions.

My goal is to use historical page-level signals to prioritize which content pages deserve human review and possible refresh.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
## 2. Fields: feature / label / context / excluded

### Features

The initial features I plan to use are:

- `impressions_90d` — historical search impressions.
- `sessions_90d` — historical sessions.
- `content_age_days` — age of the content.
- `search_volume` — historical search demand, if available in the warehouse.
- `trend_pct` is deliberately excluded from the final feature set because it is related to the label.

### Label / proxy

My label proxy is:

`is_declining_label = (trend_direction == "down")`

This represents an observed downward trend. It is a proxy for identifying pages that may deserve review; it does not prove that refreshing the page will cause recovery.

### Context

I will keep:

- `content_id` — identifies the page.
- `client_id` — identifies the client and can be useful for grouping or client-level validation.
- `month` — identifies the reporting period.

These fields provide context but should not automatically become model features.

### Excluded

I will exclude `trend_direction` and `trend_pct` from the final feature set.

The reason is that the decline label is derived from `trend_direction`, so using it as a feature would leak information from the label into the model.

I will also avoid using future outcome information when creating features.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
from huggingface_hub import HfFileSystem

fs = HfFileSystem(token=HF_TOKEN)

files = fs.ls("datasets/FlyRank/internship-warehouse", detail=True)

for file in files:
    print(file)

{'name': 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance', 'size': 0, 'type': 'directory', 'tree_id': 'a9d0e87db2923894c18291294bd3818f00a5791e', 'last_commit': None}
{'name': 'datasets/FlyRank/internship-warehouse/.gitattributes', 'size': 2504, 'type': 'file', 'blob_id': 'bed0738c7eeb449bca98b5d2f33c89a1ee56349a', 'lfs': None, 'xet_hash': None, 'last_commit': None, 'security': None}
{'name': 'datasets/FlyRank/internship-warehouse/README.md', 'size': 3038, 'type': 'file', 'blob_id': 'af501786cb97002e9646649c488d8f6e4f275fab', 'lfs': None, 'xet_hash': None, 'last_commit': None, 'security': None}
{'name': 'datasets/FlyRank/internship-warehouse/dim_clients.parquet', 'size': 3377, 'type': 'file', 'blob_id': '7c5ee024976718db679c472b80e9d44114b83dfd', 'lfs': BlobLfsInfo(size=3377, sha256='****************************************************************', pointer_size=129), 'xet_hash': '****************************************************************', 'last_commit': No

In [12]:
!pip install -q duckdb huggingface_hub

In [13]:
import duckdb

con = duckdb.connect()

print("DuckDB connected successfully")

DuckDB connected successfully


In [19]:
from huggingface_hub import HfFileSystem

fs = HfFileSystem(token=HF_TOKEN)

print(fs.info(
    "datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet"
))

{'name': 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet', 'size': 145075498, 'type': 'file', 'blob_id': '3a1435ed84d54ab479adc47836f6fb29dbc58f17', 'lfs': BlobLfsInfo(size=145075498, sha256='****************************************************************', pointer_size=134), 'xet_hash': '****************************************************************', 'last_commit': None, 'security': None}


In [ ]:
### Query 1 — Grain

I am checking whether the selected data really has one row per content page per month.

If this query returns no duplicate page-month combinations, that supports my stated unit of analysis.

In [ ]:
### Query 2 — Row count and date window

This query measures the number of rows in my March 2026 slice and checks the date span represented by those observations.

I use March as a development month instead of the final June 2026 month.

In [ ]:
### Query 3 — Availability

I use `IS TRUE` explicitly to count rows where the availability flag is actually true.

This tells me how many usable observations remain in my March 2026 slice.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
## 4. Data limits

One important limitation is that the data shows historical search and content performance, but it cannot prove that a content refresh caused a future improvement.

The declining-trend label is therefore a proxy for identifying pages that may deserve review. It should not be interpreted as proof that a refresh will succeed.

Another limitation is that the available history may not represent every page equally. Some pages may have shorter or less complete histories than others.

There may also be overlap between rolling 90-day features and monthly observations. Therefore, I need to be careful not to treat repeated historical windows as independent observations.

Finally, the final month should remain a sealed test month rather than being used to develop the label or feature logic.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.